# Config
Shared settings for every notebook. Load it at the top of each notebook in a subfolder of `src/` with `%run ../config.ipynb`.

Data pipeline (`src/data/`, run in order):

| Notebook | Reads | Writes |
|---|---|---|
| 01_download_bts | BTS / NOAA websites | `data/raw/bts/YYYY_MM.parquet`, `data/raw/ghcnh/*.psv` |
| 02_nasr | `data/raw/nasr/APT_*.csv` | `data/interim/runway_ends.parquet`, `airports.parquet` |
| 03_weather | `data/raw/ghcnh/` | `data/interim/weather_{APT}.parquet` |
| 04_flights | `data/raw/bts/`, airports | `data/interim/flights_{APT}.parquet`, `hourly_flights_{APT}.parquet` |
| 05_runway_wind | runway_ends, weather | `data/interim/runway_wind_{APT}.parquet` |
| 06_build_dataset | all interim | `data/processed/{APT}_hourly.parquet` |

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

# Project root = the nearest folder above the current notebook that holds pyproject.toml
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").exists())
DATA = ROOT / "data"
RAW = DATA / "raw"
NASR_DIR = RAW / "nasr"
GHCNH_DIR = RAW / "ghcnh"
BTS_DIR = RAW / "bts"
INTERIM = DATA / "interim"
PROCESSED = DATA / "processed"
for _p in (NASR_DIR, GHCNH_DIR, BTS_DIR, INTERIM, PROCESSED):
    _p.mkdir(parents=True, exist_ok=True)

## Study airports and periods
To add an airport, add its BTS (IATA) code and its GHCNh station ID. You can find the station ID by matching the ICAO code (e.g. KEWR) in the GHCNh station list.

In [2]:
STUDY_AIRPORTS = {
    "EWR": {"ghcnh_station": "USW00014734"},
}

BTS_START = "2023-01"   # first month of airline data
BTS_END = "2026-09"     # pinned so everyone builds the same dataset; None = latest month BTS has published
TRAIN_END = "2025-12"   # months up to and including this one are training; later months are the test set

## BTS (IATA) → NASR (FAA) airport codes
BTS uses IATA codes, and NASR uses FAA location IDs. They're the same for almost every airport. These are the exceptions found in the BTS data (each checked against APT_BASE).

In [3]:
IATA_TO_FAA = {
    "PBI": "DJT",  # Palm Beach Intl, renamed President Donald J Trump Intl
    "CLD": "CRQ",  # Carlsbad McClellan-Palomar
    "FCA": "GPI",  # Kalispell Glacier Park Intl
    "SPN": "GSN",  # Saipan
    "YUM": "NYL",  # Yuma MCAS / Yuma Intl
    "AZA": "IWA",  # Phoenix-Mesa Gateway
    "MQT": "SAW",  # Marquette Sawyer
    "GUF": "JKA",  # Gulf Shores
    "SCE": "UNV",  # State College
    "USA": "JQF",  # Concord-Padgett
    "HHH": "HXD",  # Hilton Head
}
FAA_TO_IATA = {v: k for k, v in IATA_TO_FAA.items()}


def to_faa(code):
    return IATA_TO_FAA.get(code, code)

## Runway filters and wind limits

In [4]:
RUNWAY_MIN_LEN_FT = 5000
PAVED_SURFACES = ("ASPH", "CONC")
ILS_TYPES = ("ILS", "ILS/DME", "LOC/GS")   # precision approaches (LOC/GS is a localizer plus glideslope, i.e. a full ILS)

MAX_CROSSWIND_KT = 20
MAX_TAILWIND_KT = 5
CALM_WIND_KT = 3

## Weather processing

In [5]:
WX_FFILL_LIMIT_H = 2
MS_TO_KT = 1.943844
KM_TO_SM = 0.621371
M_TO_FT = 3.28084
# GHCNh quality codes meaning suspect/erroneous (docs §VI). Lowercase letters are QC-check failures.
GHCNH_BAD_QC = {"2", "3", "6", "7"}

## Label settings
An airport-hour counts as **over capacity** (`over_capacity = 1`) if either of these holds:
- the share of scheduled arrivals with NAS + weather delay ≥ `NAS_WX_DELAY_MIN` minutes is at least `NAS_WX_SHARE_MIN`. It's only computed when at least `MIN_SCHED_ARR_FOR_SHARE` arrivals are scheduled.
- at least `CANCEL_WX_NAS_MIN` flights are cancelled for weather or NAS (cancellation codes B or C)

Backlog growth is **not** part of the label. Diagnostics showed that hours flagged only by backlog growth had normal delays and clustered at schedule peaks (7–8am, 5pm), so they measured how the schedule bunches flights, not capacity. Backlog is still kept as a feature.

`LABEL_SENSITIVITY` lists the alternative thresholds that 06 reports, so you can check the results hold up.

In [6]:
NAS_WX_DELAY_MIN = 15
NAS_WX_SHARE_MIN = 0.25
MIN_SCHED_ARR_FOR_SHARE = 3
CANCEL_WX_NAS_MIN = 3

LABEL_SENSITIVITY = {
    "strict":  dict(share=0.40, cancel=5),
    "default": dict(share=NAS_WX_SHARE_MIN, cancel=CANCEL_WX_NAS_MIN),
    "loose":   dict(share=0.15, cancel=2),
    "nas_only": dict(share=NAS_WX_SHARE_MIN, cancel=np.inf),   # FAA-attributed delay alone
}

HORIZONS = [1, 3, 6]             # predict over capacity at hour t+H for each H

OPS_DAY_START_LOCAL_HOUR = 4     # backlog resets at 04:00 local time
DEMONSTRATED_WINDOW = "28D"     # rolling window for demonstrated capacity
DEMONSTRATED_Q = 0.95

In [7]:
print(f"ROOT = {ROOT}")
print(f"study airports: {list(STUDY_AIRPORTS)} | BTS {BTS_START} → {BTS_END or 'latest'} | train ≤ {TRAIN_END}")

ROOT = /Users/dylannegvesky/Documents/FSU Documents/Fall 2026/ProjDS/PDS_Final
study airports: ['EWR'] | BTS 2023-01 → 2026-09 | train ≤ 2025-12
